# SkinSense AI — Day 1: Dataset Setup + Quick Pipeline Test
**Member 2 — ML Engineer**

Goal for today: get a working data pipeline + a model that trains and saves, even with low accuracy. Full training happens Day 2.

Classes: `melanoma`, `nevus`, `basal_cell_carcinoma`, `eczema`, `normal`

Sources combined (no single Kaggle dataset has exactly these 5 classes):
- HAM10000 → melanoma, nevus, basal cell carcinoma
- DermNet → eczema
- Normal skin dataset → normal

## 1. Setup Kaggle API
Fill in your Kaggle username and API key below, then run this cell. (Your username is visible in your Kaggle profile URL: kaggle.com/yourusername)

In [ ]:
!pip install -q kaggle

import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_API_KEY"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready.')

## 2. Download datasets

In [ ]:
!mkdir -p /content/raw
%cd /content/raw

# HAM10000: melanoma, nevus, basal cell carcinoma
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q -o skin-cancer-mnist-ham10000.zip -d ham10000

# DermNet: eczema
!kaggle datasets download -d shubhamgoel27/dermnet
!unzip -q -o dermnet.zip -d dermnet

# Normal skin
!kaggle datasets download -d ahdasdwdasd/our-normal-skin
!unzip -q -o our-normal-skin.zip -d normal_skin

print('Downloads complete.')

**Note:** if any `unzip` command fails because the filename inside the zip differs, run `!ls /content/raw` after each download to check the actual file/folder name and adjust the path below. Kaggle dataset internal structures occasionally change.

## 3. Organize into 5 class folders
This builds `/content/dataset/<class_name>/*.jpg` for each of the 5 classes.

In [ ]:
import os, shutil, glob
import pandas as pd

BASE = '/content/dataset'
CLASSES = ['melanoma', 'nevus', 'basal_cell_carcinoma', 'eczema', 'normal']
for c in CLASSES:
    os.makedirs(f'{BASE}/{c}', exist_ok=True)

# --- HAM10000: melanoma (mel), nevus (nv), basal cell carcinoma (bcc) ---
meta = pd.read_csv('/content/raw/ham10000/HAM10000_metadata.csv')
ham_img_dirs = glob.glob('/content/raw/ham10000/HAM10000_images_part_*')

dx_to_class = {'mel': 'melanoma', 'nv': 'nevus', 'bcc': 'basal_cell_carcinoma'}

# build a lookup of image_id -> full path across both image part folders
img_lookup = {}
for d in ham_img_dirs:
    for fp in glob.glob(f'{d}/*.jpg'):
        img_id = os.path.splitext(os.path.basename(fp))[0]
        img_lookup[img_id] = fp

copied = {k: 0 for k in dx_to_class.values()}
for _, row in meta.iterrows():
    dx = row['dx']
    if dx in dx_to_class:
        img_id = row['image_id']
        src = img_lookup.get(img_id)
        if src:
            dst_class = dx_to_class[dx]
            shutil.copy(src, f'{BASE}/{dst_class}/{img_id}.jpg')
            copied[dst_class] += 1
print('HAM10000 copied:', copied)

# --- DermNet: eczema ---
# DermNet folder names vary slightly by upload version, e.g. 'Eczema Photos'
eczema_dirs = glob.glob('/content/raw/dermnet/**/*czema*', recursive=True)
eczema_dirs = [d for d in eczema_dirs if os.path.isdir(d)]
print('Found eczema folders:', eczema_dirs)

count = 0
for d in eczema_dirs:
    for fp in glob.glob(f'{d}/*'):
        if fp.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(fp, f"{BASE}/eczema/{count}_{os.path.basename(fp)}")
            count += 1
print('Eczema images copied:', count)

# --- Normal skin ---
normal_files = glob.glob('/content/raw/normal_skin/**/*', recursive=True)
normal_files = [f for f in normal_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print('Found normal skin images:', len(normal_files))

count = 0
for fp in normal_files:
    shutil.copy(fp, f"{BASE}/normal/{count}_{os.path.basename(fp)}")
    count += 1
print('Normal images copied:', count)

# --- Summary ---
for c in CLASSES:
    n = len(glob.glob(f'{BASE}/{c}/*'))
    print(f'{c}: {n} images')

**Check the printed counts.** If any class has very few images (e.g. < 200), flag it — Day 2 we may need to balance classes with augmentation or trim oversized ones. It's normal at this stage if counts are uneven; HAM10000 in particular is melanoma/nevus-heavy.

## 4. Train / Validation / Test split
80% train, 10% val, 10% test, per class.

In [ ]:
!pip install -q split-folders
import splitfolders

splitfolders.ratio(
    '/content/dataset',
    output='/content/split',
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)
print('Split complete. Folders: /content/split/train, /val, /test')

## 5. Quick pipeline test — MobileNetV2 transfer learning
This is a SHORT run (a few epochs) just to confirm the whole pipeline works end-to-end. Don't worry about accuracy today — full training with more epochs and tuning happens Day 2.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.15
).flow_from_directory('/content/split/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    '/content/split/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')

# IMPORTANT for Member 1 (backend): this is the class index order the model learns.
# It MUST be used to map prediction indices back to disease names.
print('Class index mapping (give this to Member 1):')
print(train_gen.class_indices)

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # freeze for the quick test; Day 2 we can fine-tune

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')  # 5 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Quick run: 3 epochs just to prove the pipeline trains without errors.
history = model.fit(train_gen, validation_data=val_gen, epochs=3)

## 6. Save the model
This is your Day 1 deliverable proof — a model file that loads and predicts. Day 2 we retrain properly and overwrite this file.

In [ ]:
model.save('/content/skin_model.h5')
print('Saved skin_model.h5')

from google.colab import files
files.download('/content/skin_model.h5')

## Day 1 checklist
- [ ] Dataset downloaded and organized into 5 class folders
- [ ] Train/val/test split created
- [ ] Quick model trained without errors
- [ ] `skin_model.h5` saved and downloaded
- [ ] Class index mapping noted and shared with Member 1 (printed above)
- [ ] Push this notebook + class index mapping to `model/` folder in GitHub

## Contract for Member 1 (backend)
- **Input size:** 224x224 RGB
- **Normalization:** pixel values scaled to 0–1 (divide by 255) before feeding to model
- **Class order:** exactly as printed by `train_gen.class_indices` above — paste that dict into your handoff notes, don't assume alphabetical order